In [23]:
import numpy as np 
import pandas as pd 
import re
import string
import pickle

In [24]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
        return text

In [25]:
with open('../static/model/model.pickle','rb') as f:
    model = pickle.load(f)

In [26]:
with open('../static/model/corpora/stopwords/english','r') as file:
    sw = file.read().splitlines()

In [27]:
vocab = pd.read_csv('../static/model/vocabulary.txt',header = None)
tokens = vocab[0].tolist()

In [28]:
!pip install nltk

In [29]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [30]:
def preprocessing(text):
    data = pd.DataFrame([text],columns=['tweet'])
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"]=data['tweet'].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','', x, flags=re.MULTILINE) for x in x.split()))
    data["tweet"] = data["tweet"].apply(remove_punctuations)   
    data["tweet"] = data['tweet'].str.replace(r'\d+', '', regex=True)
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(w for w in x.split() if w not in sw))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["tweet"]

In [31]:
preprocessed_txt = preprocessing(txt)

In [32]:
import numpy as np

def vectorizer(ds, vocabulary):
    vectorized_lst = []

    for sentence in ds:
        sentence_vec = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_vec[i] = 1

        vectorized_lst.append(sentence_vec)

    return np.asarray(vectorized_lst, dtype=np.float32)

In [33]:
vectorized_txt = vectorizer(preprocessed_txt,tokens)

In [34]:
model.predict(vectorized_txt)

array([1])

In [36]:
def get_prediction(vectorized_text):
    prediction = model.predict(vectorized_text)
    if prediction == 1:
        return 'negative'
    else:
        return 'positive'

In [37]:
txt = "awesome product. i love it"
preprocessed_txt = preprocessing(txt)
vectorized_txt = vectorizer(preprocessed_txt, tokens)
prediction = get_prediction(vectorized_txt)
prediction

'positive'